# gr00t one step planning deployment
### they use GR1 as example, so we do not know much details about Droid.
- image size(1,256,256,3) | uint8| padding and resize, like pi0
- For droid, ext1 is left, ext2 is right
- they have already binarized gripper action. action=1 means close, action=0 means open
- rotation is 'rpy' euler angle
- TODO: find the correct scaling factors for action!





### install pyzed
https://www.stereolabs.com/docs/development/python/install

(1) install ZED SDK

(2)
```cd "/usr/local/zed/" python3 get_python_api.py```

- In MPK, use ```conda activate gr00t``` or select env upper right

In [ ]:
import os
import torch
import gr00t

from gr00t.data.dataset import LeRobotSingleDataset
from gr00t.model.policy import Gr00tPolicy
import numpy as np

# set config

In [ ]:
# change the following paths
MODEL_PATH = "nvidia/GR00T-N1.5-3B"

# REPO_PATH is the path of the pip install gr00t repo and one level up
REPO_PATH = os.path.dirname(os.path.dirname(gr00t.__file__))
DATASET_PATH = os.path.join(REPO_PATH, "demo_data/robot_sim.PickNPlace")
EMBODIMENT_TAG = "oxe_droid"

device = "cuda" if torch.cuda.is_available() else "cpu"

# load model

In [ ]:
from gr00t.experiment.data_config import DATA_CONFIG_MAP

# can add [optional] denoising step in policy

data_config = DATA_CONFIG_MAP["oxe_droid"]
modality_config = data_config.modality_config()
modality_transform = data_config.transform()

policy = Gr00tPolicy(
    model_path=MODEL_PATH,
    embodiment_tag=EMBODIMENT_TAG,
    modality_config=modality_config,
    modality_transform=modality_transform,
    device=device,
)

# print out the policy model architecture
print(policy.model)

### load image preprocess functions

In [ ]:
from PIL import Image
def resize_with_pad(images: np.ndarray, height: int, width: int, method=Image.BILINEAR) -> np.ndarray:
    """Replicates tf.image.resize_with_pad for multiple images using PIL. Resizes a batch of images to a target height.

    Args:
        images: A batch of images in [..., height, width, channel] format.
        height: The target height of the image.
        width: The target width of the image.
        method: The interpolation method to use. Default is bilinear.

    Returns:
        The resized images in [..., height, width, channel].
    """
    # If the images are already the correct size, return them as is.
    if images.shape[-3:-1] == (height, width):
        return images

    original_shape = images.shape

    images = images.reshape(-1, *original_shape[-3:])
    resized = np.stack([_resize_with_pad_pil(Image.fromarray(im), height, width, method=method) for im in images])
    return resized.reshape(*original_shape[:-3], *resized.shape[-3:])


def _resize_with_pad_pil(image: Image.Image, height: int, width: int, method: int) -> Image.Image:
    """Replicates tf.image.resize_with_pad for one image using PIL. Resizes an image to a target height and
    width without distortion by padding with zeros.

    Unlike the jax version, note that PIL uses [width, height, channel] ordering instead of [batch, h, w, c].
    """
    cur_width, cur_height = image.size
    if cur_width == width and cur_height == height:
        return image  # No need to resize if the image is already the correct size.

    ratio = max(cur_width / width, cur_height / height)
    resized_height = int(cur_height / ratio)
    resized_width = int(cur_width / ratio)
    resized_image = image.resize((resized_width, resized_height), resample=method)

    zero_image = Image.new(resized_image.mode, (width, height), 0)
    pad_height = max(0, int((height - resized_height) / 2))
    pad_width = max(0, int((width - resized_width) / 2))
    zero_image.paste(resized_image, (pad_width, pad_height))
    assert zero_image.size == (width, height)
    return zero_image

### initial realsense cameras

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt



class RealSenseCamera:
    def __init__(self, serial_number, width=320, height=180, fps=30):
        self.serial = serial_number
        self.pipeline = rs.pipeline()
        self.config = rs.config()
        self.config.enable_device(self.serial)
        self.config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)
        self.pipeline.start(self.config)


    def get_image(self):
        frames = self.pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            return None

        bgr = np.asanyarray(color_frame.get_data())
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

        return rgb

    def release(self):
        self.pipeline.stop()



# change the following serial numbers to your own cameras
serial_left = "TBD"
serial_right = "TBD"

# initialize cameras
cam_left = RealSenseCamera(serial_left, width=320, height=180, fps=30)
cam_right = RealSenseCamera(serial_right, width=320, height=180, fps=30)

### initial wrist camera

In [ ]:
import pyzed.sl as sl
import numpy as np
import cv2

class ZEDMiniCamera:
    def __init__(self, resolution=sl.RESOLUTION.HD720, fps=30, depth=False):
        print("[INFO] Initializing ZED Mini camera")
        self.zed = sl.Camera()

        init_params = sl.InitParameters()
        init_params.camera_resolution = resolution
        init_params.camera_fps = fps
        init_params.depth_mode = sl.DEPTH_MODE.PERFORMANCE if depth else sl.DEPTH_MODE.NONE
        init_params.coordinate_units = sl.UNIT.MILLIMETER  # For depth, if used

        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            raise RuntimeError(f"[ERROR] Failed to open ZED camera: {status}")
        print("[SUCCESS] ZED camera opened successfully")

        self.image = sl.Mat()

    def get_image(self):
        if self.zed.grab() == sl.ERROR_CODE.SUCCESS:
            self.zed.retrieve_image(self.image, sl.VIEW.LEFT)  # LEFT image (color)
            bgr_image = self.image.get_data()
            rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
            return rgb_image
        else:
            print("[WARN] Failed to grab frame from ZED")
            return None
    
    def release(self):
        self.zed.close()
        print("[INFO] ZED camera released")

cam_wrist = ZEDMiniCamera(resolution=sl.RESOLUTION.VGA, fps=30)

### get current observation

In [ ]:
# Resize images to 256x256 with padding
img_left = resize_with_pad(cam_left.get_image(), 256, 256)
img_wrist = resize_with_pad(cam_wrist.get_image(), 256, 256)
img_right = resize_with_pad(cam_right.get_image(), 256, 256)

# combine the two images
combined = np.hstack((img_left, img_wrist, img_right))  # shape (256, 768, 3)

# display the combined image
plt.figure(figsize=(8, 4))
plt.imshow(combined)
plt.title("Left + Wrist + Right ")
plt.axis('off')
plt.show()

### release cameras

In [ ]:
cam_left.release()
cam_wrist.release()
cam_right.release()

# GR00T inference client
- note: deoxys OSC controller uses Axis-Angle; action from gr00t is Euler angles



### one step planning in deployment script
- NUM_ACTION_PROPOSAL defines how many action proposals from VLA

In [ ]:
# vla_policy_client
import Pyro5.api
import numpy as np
import time
from gr00t.data.transform.state_action import RotationTransform



euler_to_axis_angle_tform = RotationTransform(
    from_rep="euler_angles_rpy",
    to_rep="axis_angle",
)


# connect to the Pyro server
ns = Pyro5.api.locate_ns()  # Locate the name server
uri = ns.lookup("gr00t_controller")  # Look up the registered object by name
controller = Pyro5.api.Proxy(uri)


# set global variables
MAX_INFERENCE = 500
NUM_ACTION_PROPOSAL = 3  # number of action proposals to generate

# make sure the scaling factors work for World model
MAGIC_NUMBER_TRANSLATION = 0.01
MAGIC_NUMBER_ROTATION = 0.01



video_buffer = []

# prompt
prompt = "pick up the green cube and place it in the cup"  # <-- change prompt

# dummy action
action = np.zeros((16,7), dtype=np.float32)
action_list = [action.tolist() for _ in range(NUM_ACTION_PROPOSAL)] # list of actions to send to the controller
# dummy images
image_left = np.zeros((180, 320, 3), dtype=np.uint8)
image_wrist = np.zeros((180, 320, 3), dtype=np.uint8)
image_right = np.zeros((180, 320, 3), dtype=np.uint8)


for step in range(MAX_INFERENCE):

    print(f"\n=== Step {step} ===")
    data_to_send = {
        "type": "action",
        "data": action_list,
        "image_left": image_left.tolist(),   # convert to list for sending
        "image_wrist": image_wrist.tolist(),
        "image_right": image_right.tolist(),
        "prompt": prompt,
        "policy_step": step,
    }

    # send action to the controller and get new observation
    obs = controller.step(data_to_send)



    # get image from cameras
    image_left = cam_left.get_image()
    image_wrist = cv2.resize(cam_wrist.get_image(), (image_left.shape[1], image_left.shape[0])) # resize wrist image to match left image size
    image_right = cam_right.get_image()

    # preprocess images for policy
    img_left = resize_with_pad(image_left, 256, 256)[None, ...]     # shape: (1, 256, 256, 3)
    img_wrist = resize_with_pad(image_wrist, 256, 256)[None, ...]
    img_right = resize_with_pad(image_right, 256, 256)[None, ...]

    # save images to video buffer
    combined = np.hstack([img_left[0], img_right[0], img_wrist[0]])
    video_buffer.append(combined)



    step_data = {
        'video.exterior_image_1': img_left,
        'video.exterior_image_2': img_right,
        'video.wrist_image': img_wrist,
        'state.eef_position': np.array(obs['robot_pos'], dtype=np.float32),  # (1, 3)
        'state.eef_rotation': np.array(obs['robot_rot'], dtype=np.float32),  # (1, 3)
        'state.gripper_position': np.array(obs['gripper_state'], dtype=np.float32),  # (1, 1)
        'annotation.language.language_instruction': [prompt]
    }

    for i in range(NUM_ACTION_PROPOSAL):
        predicted_action = policy.get_action(step_data)
        # TODO: find the correct scaling factors!!!!!!!!!!!!!!
        pos = predicted_action['action.eef_position_delta'] * MAGIC_NUMBER_TRANSLATION # (16, 3)
        rot = predicted_action['action.eef_rotation_delta'] * MAGIC_NUMBER_ROTATION  # (16, 3)

        # this scaling factors below don't work for World model
        # pos = (predicted_action["action.eef_position_delta"] / np.linalg.norm(predicted_action["action.eef_position_delta"], axis=1).reshape((-1, 1))) * 0.0125
        # rot = (predicted_action["action.eef_rotation_delta"] / np.linalg.norm(predicted_action["action.eef_rotation_delta"], axis=1).reshape((-1, 1))) * 0.2
        grip = predicted_action['action.gripper_position']  # (16,)
        if grip.ndim == 1:
            grip = grip[:, np.newaxis]

        # action_concat = np.concatenate([pos, rot, grip], axis=-1)
        rot_aa = euler_to_axis_angle_tform.forward(torch.from_numpy(rot))
        rot_aa = rot_aa.numpy()  # convert to numpy array
        # rot_aa = np.zeros_like(rot_aa)
        # pos = np.zeros_like(pos)  # zero out position for testing
        # pos[:, 0] = 0.01
        action_concat = np.concatenate([pos, rot_aa, grip], axis=-1)

        one_action_chunk = action_concat.tolist()  # convert to list for sending
        action_list[i] = one_action_chunk

### save and play gif

In [ ]:
import imageio
import cv2
from IPython.display import Image as IPyImage

logdir = "./one_step_planning_real"
os.makedirs(logdir, exist_ok=True)

gif_path = os.path.join(logdir, "test.gif")
imageio.mimsave(gif_path, video_buffer, duration=0.5)
print(f"GIF saved:{gif_path}")
IPyImage(filename=gif_path)

### this cell just used for debugging

In [ ]:
# vla_policy_client
import Pyro5.api
import numpy as np
import time
from gr00t.data.transform.state_action import RotationTransform
import cv2



euler_to_axis_angle_tform = RotationTransform(
    from_rep="euler_angles_rpy",
    to_rep="axis_angle",
)


# connect to the Pyro server
uri = "PYRO:obj_9a720a97702c4502821c9e249643b406@a100-st-p4de24xlarge-245:43029"  # copy the URI here
controller = Pyro5.api.Proxy(uri)


# set global variables
MAX_INFERENCE = 500
NUM_ACTION_PROPOSAL = 3  # number of action proposals to generate

# make sure the scaling factors work for World model
MAGIC_NUMBER_TRANSLATION = 0.01
MAGIC_NUMBER_ROTATION = 0.01



video_buffer = []

# prompt
prompt = "pick up the red cube and lift it up."  # <-- change prompt

# dummy action
action = np.zeros((16,7), dtype=np.float32)
action_list = [action.tolist() for _ in range(NUM_ACTION_PROPOSAL)] # list of actions to send to the controller
# dummy images
image_left = np.zeros((180, 320, 3), dtype=np.uint8)
image_wrist = np.zeros((180, 320, 3), dtype=np.uint8)
image_right = np.zeros((180, 320, 3), dtype=np.uint8)


for step in range(MAX_INFERENCE):

    print(f"\n=== Step {step} ===")
    data_to_send = {
        "type": "action",
        "data": action_list,
        "image_left": image_left.tolist(),   # convert to list for sending
        "image_wrist": image_wrist.tolist(),
        "image_right": image_right.tolist(),
        "prompt": prompt,
        "policy_step": step,
    }

    # send action to the controller and get new observation
    obs = controller.step(data_to_send)



    # get image from cameras
    image_left  = cv2.cvtColor(cv2.imread("./image_left.png"), cv2.COLOR_BGR2RGB)
    image_wrist = cv2.resize(cv2.cvtColor(cv2.imread("./image_wrist.png"), cv2.COLOR_BGR2RGB), (image_left.shape[1], image_left.shape[0]))
    image_right = cv2.resize(cv2.cvtColor(cv2.imread("./image_right.png"), cv2.COLOR_BGR2RGB), (image_left.shape[1], image_left.shape[0]))

    # preprocess images for policy
    img_left = resize_with_pad(image_left, 256, 256)[None, ...]     # shape: (1, 256, 256, 3)
    img_wrist = resize_with_pad(image_wrist, 256, 256)[None, ...]
    img_right = resize_with_pad(image_right, 256, 256)[None, ...]

    # save images to video buffer
    combined = np.hstack([img_left[0], img_right[0], img_wrist[0]])
    video_buffer.append(combined)



    step_data = {
        'video.exterior_image_1': img_left,
        'video.exterior_image_2': img_right,
        'video.wrist_image': img_wrist,
        'state.eef_position': np.array(obs['robot_pos'], dtype=np.float32),  # (1, 3)
        'state.eef_rotation': np.array(obs['robot_rot'], dtype=np.float32),  # (1, 3)
        'state.gripper_position': np.array(obs['gripper_state'], dtype=np.float32),  # (1, 1)
        'annotation.language.language_instruction': [prompt]
    }

    for i in range(NUM_ACTION_PROPOSAL):
        predicted_action = policy.get_action(step_data)
        # TODO: find the correct scaling factors!!!!!!!!!!!!!!

        pos = predicted_action['action.eef_position_delta'] * MAGIC_NUMBER_TRANSLATION # (16, 3)
        rot = predicted_action['action.eef_rotation_delta'] * MAGIC_NUMBER_ROTATION  # (16, 3)
        # this scaling factors below don't work for World model
        # pos = (predicted_action["action.eef_position_delta"] / np.linalg.norm(predicted_action["action.eef_position_delta"], axis=1).reshape((-1, 1))) * 0.0125
        # rot = (predicted_action["action.eef_rotation_delta"] / np.linalg.norm(predicted_action["action.eef_rotation_delta"], axis=1).reshape((-1, 1))) * 0.2
        
        grip = predicted_action['action.gripper_position']  # (16,)
        if grip.ndim == 1:
            grip = grip[:, np.newaxis]

        # action_concat = np.concatenate([pos, rot, grip], axis=-1)
        rot_aa = euler_to_axis_angle_tform.forward(torch.from_numpy(rot))
        rot_aa = rot_aa.numpy()  # convert to numpy array
        # rot_aa = np.zeros_like(rot_aa)
        # pos = np.zeros_like(pos)  # zero out position for testing
        # pos[:, 0] = 0.01
        action_concat = np.concatenate([pos, rot_aa, grip], axis=-1)

        one_action_chunk = action_concat.tolist()  # convert to list for sending
        action_list[i] = one_action_chunk